# Music Annotation Project V8.8
## Notebook 52F: Final Observable-Builder Regression and SAFE_REMOVE Replay

Fail-closed approval test before incorporating the tested observable builder into `v8_8_operator.py`.

**PASS requires:** authoritative Notebook 50 lineage; exact locked predictor schema; exact Notebook 49 candidate population; full-row predictor equivalence; exact SAFE_REMOVE replay. No clean/truth field is used as a runtime predictor.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pandas numpy pyarrow opencv-python-headless scikit-learn joblib

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil
import cv2, numpy as np, pandas as pd
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
N50_OVERRIDE=None; N49_OVERRIDE=None
N50_ROOT=ROOT/'observable_residual_mask_m5_v8_7'
N49_ROOT=ROOT/'observed_print_classifier_v8_7_notebook49'
OUT_ROOT=ROOT/'v8_8_final_builder_regression_52f'
OUT=OUT_ROOT/(datetime.now().strftime('%Y%m%d_%H%M%S')+'_52f_final_builder_regression')
T=OUT/'tables'; A=OUT/'audit'; U=OUT/'upload'; I=OUT/'inputs'
for p in [OUT,T,A,U,I]: p.mkdir(parents=True,exist_ok=False)
ATOL=1e-6; RTOL=1e-7; REMOVE_T=0.995; REGION_MEAN=0.995; REGION_FRAC=1.0; MIN_RUN=2

def require(x,msg):
    if not bool(x): raise RuntimeError(msg)
def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def readj(p): return json.loads(Path(p).read_text())
def atomj(p,x):
    q=Path(str(p)+'.tmp'); q.write_text(json.dumps(x,indent=2,sort_keys=True,default=str)); q.replace(p)
def latest(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
print('Output:',OUT)

## 1. Resolve authoritative Notebook 50 and Notebook 49

In [ ]:
N50_REQ=['experiment_identity.json','notebook50_verdict.json','selected_models.json','inputs/predictor_schema.json','tables/all_m3_oof_predictions_and_decisions.parquet']
N50=Path(N50_OVERRIDE) if N50_OVERRIDE else latest(N50_ROOT,N50_REQ)
require(N50 is not None and all((N50/r).exists() for r in N50_REQ),'No complete Notebook 50 run found.')
verdict50=readj(N50/'notebook50_verdict.json'); schema50=readj(N50/'inputs/predictor_schema.json')
verdict=str(verdict50.get('verdict',verdict50.get('decision',verdict50.get('status',''))))
require(verdict=='M5_REMOVE_ONLY_READY_FOR_NOTEBOOK51_CROSSOVER',f'Unexpected Notebook 50 verdict: {verdict}')
P=schema50.get('ordered_predictors',schema50.get('predictors',schema50.get('ordered_features',[])))
require(P,'Empty Notebook 50 predictor schema.')
N49_REQ=['03_truth_unlock/pixel_oracle_gap.parquet','01_inputs/source_record.json','02_observable_lock/candidate_lock.json']
N49=Path(N49_OVERRIDE) if N49_OVERRIDE else latest(N49_ROOT,N49_REQ)
require(N49 is not None and all((N49/r).exists() for r in N49_REQ),'No complete Notebook 49 run found.')
source49=readj(N49/'01_inputs/source_record.json')
V75=Path(source49['v75']); V82=Path(source49['source_v82']); V84=Path(source49['source_v84'])
require(V75.exists() and V82.exists() and V84.exists(),'Notebook 49 source lineage is incomplete.')
atomj(I/'source_manifest.json',{'notebook50':str(N50),'notebook49':str(N49),'v75':str(V75),'v82':str(V82),'v84':str(V84),'ordered_predictors':P})
print('Notebook 50:',N50); print('Notebook 49:',N49); print('Predictors:',len(P),P)

## 2. Load golden population and exact source arrays

In [ ]:
ref=pd.read_parquet(N49/'03_truth_unlock/pixel_oracle_gap.parquet').sort_values(['source_group','page','y','x']).reset_index(drop=True)
require(not ref.duplicated(['page','y','x']).any(),'Duplicate golden page/y/x rows.')
require(set(P).issubset(ref.columns),f'Golden table missing predictors: {sorted(set(P)-set(ref.columns))}')
status=pd.read_csv(V75/'tables/cache_status.csv'); cache={str(r.page):Path(r.cache) for _,r in status.iterrows()}
p82=pd.read_parquet(V82/'tables/all_eligible_oof_predictions.parquet')
v84o=pd.read_parquet(V84/'tables/04_oof_safety_and_value_predictions.parquet'); v84p=pd.read_parquet(V84/'tables/02_candidate_group_pixels.parquet')
if 'arm' in v84o.columns: v84o=v84o[v84o.arm.astype(str)=='multiscale_33_97'].copy()
print('Golden rows:',len(ref),'pages:',ref.page.nunique(),'groups:',ref.source_group.nunique())

## 3. Canonical observable builder

In [ ]:
def frac(a,k=5): return cv2.blur(a.astype(np.float32),(k,k))
def dist_to(mask):
    if mask.any(): return cv2.distanceTransform((~mask).astype(np.uint8),cv2.DIST_L2,5)
    return np.full(mask.shape,float(max(mask.shape)),np.float32)
def paper_surface(obs,mask,sigma=45.):
    vis=(~mask)&(obs>=200); num=cv2.GaussianBlur(obs.astype(np.float32)*vis,(0,0),sigma); den=cv2.GaussianBlur(vis.astype(np.float32),(0,0),sigma); fb=cv2.GaussianBlur(obs,(0,0),sigma)
    return np.clip(np.where(den>.02,num/np.maximum(den,1e-4),fb),0,255).astype(np.uint8)
def build_page(page,gold):
    require(page in cache,f'{page}: no V7.5 cache')
    with np.load(cache[page],allow_pickle=False) as z:
        req=['mask','g80_preserve','g80_remove','g80_ambiguous','g80_horizontal','g80_vertical','g80_compact']
        require(all(k in z.files for k in req),f'{page}: cache arrays missing')
        a={k:z[k] for k in req}
    auth=a['mask']>0; shape=auth.shape
    obs=cv2.imread(str(V82/'whole_pages'/page/'observed.png'),0); require(obs is not None and obs.shape==shape,f'{page}: observed mismatch')
    m3=np.zeros(shape,bool); gy=gold.y.to_numpy(int); gx=gold.x.to_numpy(int); m3[gy,gx]=True; require(np.all(~m3|auth),f'{page}: M3 outside authority')
    pg=p82[p82.page.astype(str)==page].copy(); require({'y','x','p_print'}.issubset(pg.columns),f'{page}: V8.2 p_print unavailable')
    strict=np.zeros(shape,bool); hit=pg.decision.astype(str).eq('PRINT').to_numpy() if 'decision' in pg.columns else pg.p_print.to_numpy(float)>=.995; strict[pg.y.to_numpy(int)[hit],pg.x.to_numpy(int)[hit]]=True
    groups=set(v84o.loc[(v84o.page.astype(str)==page)&v84o.selected.astype(bool),'group_id'].astype(str)); q=v84p[(v84p.page.astype(str)==page)&v84p.group_id.astype(str).isin(groups)]
    v84=np.zeros(shape,bool)
    if len(q): v84[q.y.to_numpy(int),q.x.to_numpy(int)]=True
    protected=(a['g80_preserve']>0)|strict|v84
    paper=paper_surface(obs,auth); norm=cv2.divide(obs,np.maximum(paper,1),scale=255); edge=cv2.Canny(obs,70,170)>0; depth=cv2.distanceTransform(m3.astype(np.uint8),cv2.DIST_L2,5)
    boundary=m3&~cv2.erode(m3.astype(np.uint8),np.ones((3,3),np.uint8)).astype(bool); bd=dist_to(boundary); dp=dist_to(protected)
    names=['g80_preserve','g80_remove','g80_ambiguous','g80_horizontal','g80_vertical','g80_compact']; loc={k:frac(a[k]>0) for k in names}; loc['edge']=frac(edge); loc['dark']=frac(obs<200)
    ys,xs=np.where(m3); pmap=np.full(shape,np.nan,np.float32); pmap[pg.y.to_numpy(int),pg.x.to_numpy(int)]=pg.p_print.to_numpy(float)
    out=pd.DataFrame({'page':page,'source_group':str(gold.source_group.iloc[0]),'y':ys,'x':xs,'observed_value':obs[ys,xs],'normalised_value':norm[ys,xs],'paper_value':paper[ys,xs],'mask_depth':depth[ys,xs],'boundary_distance':bd[ys,xs],'distance_to_protected':dp[ys,xs],'edge':edge[ys,xs].astype(np.uint8),'local_edge_fraction':loc['edge'][ys,xs],'local_dark_fraction':loc['dark'][ys,xs],'preserve':(a['g80_preserve']>0)[ys,xs].astype(np.uint8),'remove':(a['g80_remove']>0)[ys,xs].astype(np.uint8),'ambiguous':(a['g80_ambiguous']>0)[ys,xs].astype(np.uint8),'horizontal':(a['g80_horizontal']>0)[ys,xs].astype(np.uint8),'vertical':(a['g80_vertical']>0)[ys,xs].astype(np.uint8),'compact':(a['g80_compact']>0)[ys,xs].astype(np.uint8),'local_preserve_fraction':loc['g80_preserve'][ys,xs],'local_remove_fraction':loc['g80_remove'][ys,xs],'local_ambiguous_fraction':loc['g80_ambiguous'][ys,xs],'local_horizontal_fraction':loc['g80_horizontal'][ys,xs],'local_vertical_fraction':loc['g80_vertical'][ys,xs],'local_compact_fraction':loc['g80_compact'][ys,xs],'p_print':pmap[ys,xs]})
    _,lab,_,_=cv2.connectedComponentsWithStats(m3.astype(np.uint8),8); out['component_id']=lab[ys,xs].astype(int); out['region_id']=[f'{page}:m3c{c:05d}' for c in out.component_id]
    return out
rebuilt=pd.concat([build_page(str(p),g) for p,g in ref.groupby('page',sort=False)],ignore_index=True).sort_values(['source_group','page','y','x']).reset_index(drop=True)
rebuilt.to_parquet(T/'rebuilt_observable_features.parquet',index=False); print('Rebuilt:',len(rebuilt))

## 4. Full-row regression

In [ ]:
require(len(rebuilt)==len(ref),f'Row count mismatch {len(rebuilt)} != {len(ref)}')
keys=['source_group','page','y','x']; key_exact=ref[keys].astype(str).equals(rebuilt[keys].astype(str)); require(key_exact,'Ordered identity mismatch')
rows=[]
for name in ['component_id','region_id']+list(P):
    require(name in ref.columns and name in rebuilt.columns,f'Missing comparison field {name}')
    a,b=ref[name],rebuilt[name]
    if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b):
        av=pd.to_numeric(a,errors='coerce').to_numpy(float); bv=pd.to_numeric(b,errors='coerce').to_numpy(float); bad=~np.isclose(av,bv,atol=ATOL,rtol=RTOL,equal_nan=True); exact=~(np.equal(av,bv)|(np.isnan(av)&np.isnan(bv))); d=np.abs(av-bv)
        rows.append({'field':name,'mismatches':int(bad.sum()),'exact_mismatches':int(exact.sum()),'max_abs_difference':float(np.nanmax(d)) if len(d) else 0.0})
    else:
        bad=a.astype(str).to_numpy()!=b.astype(str).to_numpy(); rows.append({'field':name,'mismatches':int(bad.sum()),'exact_mismatches':int(bad.sum()),'max_abs_difference':np.nan})
fa=pd.DataFrame(rows); fa.to_csv(T/'feature_regression_summary.csv',index=False); print(fa.to_string(index=False)); print('Total tolerance mismatches:',int(fa.mismatches.sum()))

## 5. Independent SAFE_REMOVE policy replay

In [ ]:
dec=pd.read_parquet(N50/'tables/all_m3_oof_predictions_and_decisions.parquet').sort_values(['source_group','page','y','x']).reset_index(drop=True)
require({'page','source_group','y','x','p_remove','safe_remove'}.issubset(dec.columns),'Notebook 50 decision replay fields missing')
require(dec[['page','y','x']].astype(str).equals(ref[['page','y','x']].astype(str)),'Notebook 50/49 coordinate populations differ')
def replay_remove(frame):
    asserted=np.zeros(len(frame),bool)
    for page,g in frame.groupby('page',sort=False):
        idx=g.index.to_numpy(); probs=g.p_remove.to_numpy(float); candidate=probs>=REMOVE_T; m=np.zeros((int(g.y.max())+1,int(g.x.max())+1),np.uint8); m[g.y.to_numpy(int)[candidate],g.x.to_numpy(int)[candidate]]=1
        n,lab,stats,_=cv2.connectedComponentsWithStats(m,8); lookup={(int(frame.at[i,'y']),int(frame.at[i,'x'])):i for i in idx}
        for cid in range(1,n):
            yy,xx=np.where(lab==cid); ids=[lookup[(int(y),int(x))] for y,x in zip(yy,xx) if (int(y),int(x)) in lookup]
            if not ids: continue
            p=frame.loc[ids,'p_remove'].to_numpy(float); passes=len(ids)>=MIN_RUN and float(p.mean())>=REGION_MEAN and float((p>=REMOVE_T).mean())>=REGION_FRAC
            if passes: asserted[ids]=True
    return asserted
rep=replay_remove(dec); saved=dec.safe_remove.astype(bool).to_numpy(); mismatch=saved!=rep
summary={'rows':len(dec),'saved_safe_remove':int(saved.sum()),'replayed_safe_remove':int(rep.sum()),'selection_mismatches':int(mismatch.sum())}
pd.DataFrame([summary]).to_csv(T/'safe_remove_replay_summary.csv',index=False); print(summary)

## 6. Fail-closed approval verdict and upload bundle

In [ ]:
checks={'notebook50_remove_only':verdict=='M5_REMOVE_ONLY_READY_FOR_NOTEBOOK51_CROSSOVER','row_count_exact':len(rebuilt)==len(ref),'ordered_identity_exact':key_exact,'predictor_count_matches_frozen_schema':len(P)>0,'component_and_predictors_within_tolerance':int(fa.mismatches.sum())==0,'safe_remove_replay_exact':int(mismatch.sum())==0}
first=next((k for k,v in checks.items() if not v),None); verdict52f='PASS_APPROVED_FOR_V8_8_OPERATOR_BUILD' if first is None else 'FAIL_DO_NOT_BUILD_V8_8_OPERATOR'
record={'created_utc':datetime.now(timezone.utc).isoformat(),'verdict':verdict52f,'first_failed_gate':first,'checks':checks,'notebook50':str(N50),'notebook49':str(N49),'reference_rows':len(ref),'pages':int(ref.page.nunique()),'source_groups':int(ref.source_group.nunique()),'ordered_predictors':P,'feature_mismatches':int(fa.mismatches.sum()),'safe_remove_selection_mismatches':int(mismatch.sum()),'meaning':'PASS approves extraction of this tested observable builder into v8_8_operator.py; it does not authorize production.'}
atomj(OUT/'notebook52f_verdict.json',record)
summary_text='\n'.join(['NOTEBOOK 52F FINAL BUILDER REGRESSION','',f'VERDICT: {verdict52f}',f'First failed gate: {first}',f'Rows: {len(ref)}',f'Pages: {ref.page.nunique()}',f'Source groups: {ref.source_group.nunique()}',f'Predictors: {len(P)}',f'Feature mismatches: {int(fa.mismatches.sum())}',f'SAFE_REMOVE saved/replayed: {int(saved.sum())}/{int(rep.sum())}',f'SAFE_REMOVE mismatches: {int(mismatch.sum())}','','CHECKS']+[f'- {k}: {v}' for k,v in checks.items()])
(OUT/'executive_summary.txt').write_text(summary_text); print(summary_text)
files=[OUT/'notebook52f_verdict.json',OUT/'executive_summary.txt',T/'feature_regression_summary.csv',T/'safe_remove_replay_summary.csv',I/'source_manifest.json']
for p in files: shutil.copy2(p,U/p.name)
pd.DataFrame([{'file':p.name,'bytes':p.stat().st_size,'sha256':sha256_file(p)} for p in sorted(U.iterdir()) if p.is_file()]).to_csv(U/'upload_manifest.csv',index=False)
(U/'README.txt').write_text('Notebook 52F final observable-builder regression package. PASS approves builder extraction into v8_8_operator.py; not production authorisation.\n')
archive=shutil.make_archive(str(U/'notebook52f_upload_bundle'),'zip',root_dir=U); print('Upload:',archive)
if first: raise RuntimeError(f'52F failed at {first}; do not build v8_8_operator.py.')